# Proyecto KDD y ETL\n\n**Integrantes:** Manuel Mora (100065865), Stephany Ángeles (100063069), Erick Reynoso Torres (100068266) y Enmanuel Jiménez (100066650).\n\n**Persona D · soporte e integración:** Manuel Mora.\n\nEjecuta primero el pipeline y después las celdas en orden. Para guardar automáticamente todas las salidas visibles en este mismo notebook se puede usar `python src/persona_d_entrega.py`. Las métricas y figuras también quedan registradas en `evidencias/`.\n

In [ ]:
from pathlib import Path
import sys, sqlite3, json, pandas as pd
candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for p in candidates if (p/'src/pipeline.py').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Abre Jupyter desde la carpeta del proyecto o notebooks.')
db = ROOT/'data/proyecto_kdd.sqlite'
if not db.is_file():
    raise FileNotFoundError('Primero ejecuta: python src/pipeline.py --stage all')
sys.path.insert(0, str(ROOT/'src'))
import pipeline
conn = sqlite3.connect(db.resolve().as_uri() + '?mode=ro', uri=True)
print('Conexión SQL establecida en modo solo lectura')


## Selección y validación inicial

In [ ]:
raw = pd.read_sql_query('SELECT * FROM bronze_retail',conn)
assert len(raw)>100000
print('Dimensiones:', raw.shape)
print(raw.drop(columns=['CustomerID','InvoiceNo'], errors='ignore').head().to_string(index=False))
print(json.loads((ROOT/'evidencias/validacion_bronze.json').read_text()))

## Limpieza con Pandas
La función siguiente se ejecuta con `python src/pipeline.py --stage silver`. Se muestra el código exacto y se verifica su salida SQL sin recargar la base durante la lectura del notebook.

In [ ]:
import inspect
print(inspect.getsource(pipeline.silver))

In [ ]:
silver = pd.read_sql_query('SELECT * FROM silver_retail', conn)
rejected = pd.read_sql_query('SELECT reject_reason,COUNT(*) AS filas FROM silver_retail_rechazos GROUP BY reject_reason',conn)
print(rejected.to_string(index=False))
assert len(raw)==len(silver)+int(rejected.filas.sum())
print('Conciliación Bronze/Silver correcta:',len(raw),len(silver))

## Gold y transformaciones

In [ ]:
print(inspect.getsource(pipeline.gold))
print(pd.read_sql_query('SELECT * FROM gold_retail_mensual LIMIT 10',conn).to_string(index=False))

## Minería reproducible
Cortes temporales, objetivos futuros, transformaciones ajustadas sólo con entrenamiento y modelos con semilla 42.

In [ ]:
print(inspect.getsource(pipeline.snapshots))
print(inspect.getsource(pipeline.mining))

In [ ]:
metrics=json.loads((ROOT/'evidencias/metricas.json').read_text())
print(json.dumps(metrics,indent=2,ensure_ascii=False))

## Gráficos producidos por la ejecución

![retail_tendencia.png](../evidencias/retail_tendencia.png)

![retail_distribucion.png](../evidencias/retail_distribucion.png)

![matriz_confusion.png](../evidencias/matriz_confusion.png)

![roc.png](../evidencias/roc.png)

![regresion.png](../evidencias/regresion.png)

![clusters.png](../evidencias/clusters.png)

![productos_eda.png](../evidencias/productos_eda.png)

## Ejecución completa opcional
Para reconstruir todas las capas, ejecutar desde la raíz: `python src/pipeline.py --stage all`. Para regenerar CSV originales primero ejecutar `--stage extract`. No actualizar Power BI hasta que la verificación termine correctamente.

In [ ]:
print(json.loads((ROOT/'evidencias/verificacion.json').read_text()))
conn.close()